# Imports

In [77]:
import sys
import pandas as pd
import os
import numpy as np
import plotly.io as pio
import plotly.express as px
import string
import re
import seaborn as sns
sys.path.append('..')
from plotly.graph_objs import Figure
from collections import defaultdict
from functools import reduce
from definitions import ROOT_DIR, DATA_DIR

# Utils

In [2]:
FIGURES_DIR = os.path.join(ROOT_DIR, 'assets', 'corpus_statistics')
os.makedirs(FIGURES_DIR, exist_ok=True)

In [3]:
pio.templates.default = "seaborn"

In [4]:
def save_fig(fig : Figure, filename : str) -> None:
    fig.write_image(os.path.join(FIGURES_DIR, filename))

def show(fig : Figure) -> None:
    fig.update_layout(font_size = 12)
    fig.show()

# Corpus

- Load the data

In [34]:
config = 'ls2n-duke'
data_path = os.path.join(DATA_DIR, config, 'keyword_extraction', 'data.jsonl')
data = pd.read_json(data_path, lines=True)

In [35]:
data.head()

,id,title,abstract,keywords,authors,venues,teams,date,labels
0,5471943,Assessing Graph Neural Networks for latency an...,Accurately estimating the latency and power co...,"[multicore, power consumption, latency estimat...","[Oscar Roussel, Zainab Ghrayeb, Sébastien Le N...","[spi, spi.other, spi.tron]",[LS2N-DUKE],2026-04-22 00:00:00+00:00,"[Industrial Digitalization, Digital Engineerin..."
1,5477204,Graph Neural Networks for Graph-Level Regressi...,Accurately predicting the power consumption an...,"[early stage performance prediction, multi out...","[Oscar Roussel, Zainab Ghrayeb, Sébastien Le N...","[spi, spi.other, spi.tron]",[LS2N-DUKE],2026-04-22 00:00:00+00:00,"[Industrial Digitalization, Digital Engineerin..."
2,5400995,Prediction Fairness with respect to Sensitive ...,Algorithmic fairness is an important axis of e...,"[machine learning fairness, ml regularization,...","[Sajda Orellana, François Queyroi, Hoel Le Cap...","[info.info-ai, info.info-ne]",[LS2N-DUKE],2025-12-03 00:00:00+00:00,[Industrial Digitalization]
3,5106183,Machine Learning Approaches for Application Ma...,Application mapping optimization is a critical...,[application mapping network on chip noc predi...,"[Zainab Ghrayeb, Sébastien Le Nours, Christoph...","[spi.tron, spi]",[LS2N-DUKE],2025-06-11 00:00:00+00:00,"[Industrial Digitalization, Digital Engineerin..."
4,5020362,Foundation Model ColorMap : A Framework for Ex...,Recent advancements in AI have led to the deve...,"[deep learning, large language models, artific...","[Languénou Eric, Christophe Hurter]",[info],[LS2N-DUKE],2025-04-26 00:00:00+00:00,[Industrial Digitalization]


- By Year

In [ ]:
date = data['date'].apply(lambda x : int(str(x)[:4])).value_counts()
date.index <= 2015

array([False, False, False, False, False, False, False, False, False,
       False,  True])

In [44]:
date = data['date'].apply(lambda x : int(str(x)[:4])).value_counts()
v = date[date.index <= 2015].sum()
date = date[date.index > 2015]
date.index = date.index.map(str)
date.loc['<= 2015'] = v
date = date.iloc[date.index.map(lambda x : str(x)[-4:]).argsort()].reset_index()
date

,date,count
0,<= 2015,1
1,2017,16
2,2018,18
3,2019,11
4,2020,10
5,2021,22
6,2022,9
7,2023,8
8,2024,15
9,2025,5


In [45]:
fig = px.bar(
    data_frame=date,
    x="date",
    y="count",
    text = "count",
    width=800,
    height=550
)
save_fig(fig, 'year_distribution.svg')
show(fig)

- Keywords

In [46]:
keywords = reduce(lambda a,b : a + b, data['keywords'])
keywords = list(map(str.lower, keywords))
index, count = np.unique(keywords,return_counts=True)
keywords = pd.Series(index=index,data=count).sort_values(ascending=False)
print(len(keywords))

1291


In [47]:
fig = px.bar(
    data_frame = keywords.iloc[:13].sort_values(ascending=True).reset_index(),
    y = "index",
    x = 0,
    text = 0,
    width=800,
    height=450
)
fig.update_layout(xaxis_title="Frequency",yaxis_title="Keyword")
save_fig(fig, 'top_10_most_frequent_keywords.svg')
show(fig)

- Categories

In [51]:
sub_categories = reduce(lambda a,b : a + b, data['venues'])
sub_categories = pd.Series(sub_categories).value_counts()
categories = sub_categories.groupby(by=lambda x : x.split('.')[0]).sum().sort_values(ascending=False)
categories

info    172
sdv      16
spi      15
shs       9
chim      5
scco      3
stat      3
Name: count, dtype: int64

In [52]:
fig = px.bar(
    data_frame = categories.sort_values(ascending=True).reset_index(),
    y = "index",
    x = "count",
    text = "count",
    width=800,
    height=450
)
fig.update_layout(xaxis_title="Frequency",yaxis_title="Category")
save_fig(fig, 'top_10_most_frequent_categories.svg')
show(fig)

- Teams

In [54]:
teams = list(set(reduce(lambda a,b : a + b, data["teams"])))
# teams.remove('LS2N')

for team in teams:
    data[team] = data["teams"].apply(lambda x : team in x)

In [ ]:
fig = px.bar(
    data_frame = data[teams]
        .sum()
        .reset_index()
        .rename(columns = { "index" : "team", 0 : "Number Of Articles" })
        .sort_values(by = "Number Of Articles"),
    y = "Team",
    x = "Number Of Articles",
    text = "Number Of Articles",
    width = 960,
    height = 600
)

save_fig(fig, 'teams_distribution.svg')
fig.show()

- DSIs

In [64]:
data['labels'].apply(lambda x : len(x) > 0).value_counts()

labels
True     75
False    42
Name: count, dtype: int64

In [67]:
# classified Vs Not classified
px.pie(
    names = ['Labeled', 'Not labeled'],
    values = data['labels'].apply(lambda x : len(x) > 0).value_counts(),
    width = 850,
    height = 600
)

In [71]:
# Number of Labels
labels = reduce(lambda a, b : a+b, data['labels'].tolist())
labels = pd.Series(labels).value_counts()
labels.reset_index()

,index,count
0,Industrial Digitalization,58
1,Monitoring and Analysis,20
2,Digital Engineering of Products and Processes,14
3,Industrial Organization Performance,14
4,User-Centered Design,13
5,Preparation for New Jobs in the Future Industry,8
6,Augmentation of Human Capabilities in Manufact...,5
7,Composites and New Materials,3
8,Processes and Assembly,2
9,Eco-responsible Processes,1


In [75]:
fig = px.bar(
    data_frame = labels
        .sort_values()
        .reset_index()
        .rename(columns={ 'index' : 'DSI', 'count' : 'Number Of Articles' }),
    y = "DSI",
    x = "Number Of Articles",
    text = "Number Of Articles",
    width = 960,
    height = 600
)

save_fig(fig, 'dsis_distribution.svg')
fig.show()

- Team By Domain :

In [86]:
teams_labels = []

for idx, row in data.iterrows():

    for team in row['teams']:
        for label in row['labels']:
            teams_labels.append({ "team" : team, "label" : label })

teams_labels = pd.DataFrame(teams_labels)
teams_labels = teams_labels[['team', 'label']].value_counts().reset_index()

In [87]:
teams_labels

,team,label,count
0,LS2N-DUKE,Industrial Digitalization,58
1,LS2N-DUKE,Monitoring and Analysis,20
2,LS2N-DUKE,Digital Engineering of Products and Processes,14
3,LS2N-DUKE,Industrial Organization Performance,14
4,LS2N-DUKE,User-Centered Design,13
5,LS2N-DUKE,Preparation for New Jobs in the Future Industry,8
6,LS2N-DUKE,Augmentation of Human Capabilities in Manufact...,5
7,LS2N-DUKE,Composites and New Materials,3
8,LS2N-DUKE,Processes and Assembly,2
9,LS2N-DUKE,Eco-responsible Processes,1


In [88]:
fig = px.bar(
    data_frame = teams_labels,
    x = "count",
    y = "team",
    color = "label",
    width = 1200,
    height = 600,
)
fig.update_layout(barmode='stack', yaxis={'categoryorder':'total ascending'})
fig.update_layout(xaxis_title = "Number of Articles", yaxis_title = "Team")
save_fig(fig, 'teams_by_sub_domain.svg')
show(fig)

In [89]:
fig = px.bar(
    data_frame = teams_labels,
    x = "count",
    y = "label",
    color = "team",
    width = 1200,
    height = 600
)
fig.update_layout(barmode='stack', yaxis={'categoryorder':'total ascending'})
fig.update_layout(xaxis_title = "Number of Articles", yaxis_title = "Sub-Domain")
save_fig(fig, 'sub_domains_by_team.svg')
show(fig)